# Proper Baseline

Essay
 → Word2Vec embedding
 → Essay vector (average)
 → SVR regression
 → predicted score (continuous)
 → round về 0.5 band


In [1]:
import pandas as pd
import numpy as np
import random

import gensim
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import cohen_kappa_score
from scipy.stats import pearsonr

In [2]:
SEED = 42
RANDOM_SEED = 42

random.seed(SEED)
np.random.seed(SEED)


In [3]:
df = pd.read_csv("task1_cleaned_final.csv")



In [4]:
df = df.dropna(subset=[
    "Essay",
    "Task_Achievement",
    "Coherence_Cohesion",
    "Lexical_Resource",
    "Range_Accuracy"
]).reset_index(drop=True)

features = [
    "word_count",
    "unique_words",
    "ttr",
    "avg_word_len",
    "sentence_count",
    "avg_sentence_len",
    "sentence_len_var",
    "long_word_ratio",
    "short_word_ratio",
    "punct_density",
]

print("Total len:", len(df))

Total len: 8327


In [5]:
w2v_essays = df["Essay"].apply(gensim.utils.simple_preprocess)

X = w2v_essays.tolist()

y = df[[
    "Task_Achievement",
    "Coherence_Cohesion",
    "Lexical_Resource",
    "Range_Accuracy"
]].values

feat_values = df[features].values

In [6]:
idx = np.arange(len(df))

train_idx, temp_idx = train_test_split(
    idx,
    test_size=0.3,
    random_state=RANDOM_SEED,
    shuffle=True
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=2/3,
    random_state=RANDOM_SEED
)

X_train = [X[i] for i in train_idx]
X_val   = [X[i] for i in val_idx]
X_test  = [X[i] for i in test_idx]

y_train = y[train_idx]
y_val   = y[val_idx]
y_test  = y[test_idx]

feat_train = feat_values[train_idx]
feat_val   = feat_values[val_idx]
feat_test  = feat_values[test_idx]

print(len(train_idx), len(val_idx), len(test_idx))


5828 833 1666


In [7]:
scaler_feat = StandardScaler()

feat_train = scaler_feat.fit_transform(feat_train)
feat_val   = scaler_feat.transform(feat_val)
feat_test  = scaler_feat.transform(feat_test)

In [8]:
windows = [3, 5, 8]

best_score = float("inf")
best_model = None
best_window = None

logs = []


In [9]:

for w in windows:

    print(f"\n========== WINDOW = {w} ==========")

    w2v = Word2Vec(
        sentences=X_train,
        vector_size=300,
        window=w,
        min_count=1,
        workers=4,
        sg=1,
        seed=RANDOM_SEED
    )

    def get_vec(tokens):
        vecs = [w2v.wv[t] for t in tokens if t in w2v.wv]
        if len(vecs) == 0:
            return np.zeros(300)
        return np.mean(vecs, axis=0)

    X_train_vec = np.vstack([get_vec(t) for t in X_train])
    X_val_vec   = np.vstack([get_vec(t) for t in X_val])

    X_train_full = np.hstack([X_train_vec, feat_train])
    X_val_full   = np.hstack([X_val_vec, feat_val])

    scaler_tmp = StandardScaler()

    X_train_scaled = scaler_tmp.fit_transform(X_train_full)
    X_val_scaled   = scaler_tmp.transform(X_val_full)

    svr = MultiOutputRegressor(SVR(C=10, epsilon=0.1))

    svr.fit(X_train_scaled, y_train)

    y_val_pred = svr.predict(X_val_scaled)

    mae = mean_absolute_error(y_val, y_val_pred)

    print("Val MAE:", mae)

    logs.append({
        "window": w,
        "val_mae": mae
    })

    if mae < best_score:
        best_score = mae
        best_window = w
        best_w2v = w2v
        best_scaler = scaler_tmp
        best_svr = svr

print("\nBest window:", best_window)




========== WINDOW = 3 ==========
Val MAE: 0.9264564497766619

========== WINDOW = 5 ==========
Val MAE: 0.9146863208463714

========== WINDOW = 8 ==========
Val MAE: 0.9174060484248154

Best window: 5


In [10]:
log_df = pd.DataFrame(logs)
log_df.to_csv("SVR_W2V_strong_ml_logs.csv", index=False)

print("\nBest Val MAE:", best_score)


Best Val MAE: 0.9146863208463714


In [11]:
def essay_vector(tokens):
    vecs = [best_w2v.wv[t] for t in tokens if t in best_w2v.wv]
    if len(vecs) == 0:
        return np.zeros(best_w2v.vector_size)
    return np.mean(vecs, axis=0)


In [14]:
X_test_vec = np.vstack([essay_vector(t) for t in X_test])

X_test_full = np.hstack([X_test_vec, feat_test])

X_test_scaled = best_scaler.transform(X_test_full)


y_pred = best_svr.predict(X_test_scaled)
y_pred = np.clip(y_pred, 1, 9)

In [15]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
pearson = pearsonr(y_test.flatten(), y_pred.flatten())[0]

rounded_preds = np.round(y_pred * 2) / 2
within_half = np.mean(np.abs(rounded_preds - y_test) <= 0.5)

true_band = np.round(y_test.mean(axis=1) * 2).astype(int)
pred_band = np.round(y_pred.mean(axis=1) * 2).astype(int)

qwk = cohen_kappa_score(true_band, pred_band, weights="quadratic")

print("\nTEST RESULTS")
print("MAE:", mae)
print("RMSE:", rmse)
print("Pearson:", pearson)
print("Within ±0.5:", within_half)
print("QWK:", qwk)



TEST RESULTS
MAE: 0.9138303076197166
RMSE: 1.161230500207109
Pearson: 0.6913955189775173
Within ±0.5: 0.4923469387755102
QWK: 0.6817857511777127


In [ ]:
def predict_task1_svr(essay):

    tokens = gensim.utils.simple_preprocess(essay)

    vec = essay_vector(tokens)

    word_count = len(tokens)
    unique_words = len(set(tokens))
    ttr = unique_words / word_count if word_count > 0 else 0
    avg_word_len = np.mean([len(w) for w in tokens]) if word_count > 0 else 0

    sentences = essay.split(".")
    sentence_count = len([s for s in sentences if s.strip() != ""])
    avg_sentence_len = word_count / max(sentence_count,1)

    sentence_len_var = np.var([len(s.split()) for s in sentences if s.strip() != ""]) if sentence_count > 0 else 0

    long_word_ratio = np.mean([len(w)>6 for w in tokens]) if word_count > 0 else 0
    short_word_ratio = np.mean([len(w)<=3 for w in tokens]) if word_count > 0 else 0

    punct_density = len([c for c in essay if c in ",.!?;:"]) / max(word_count,1)

    feat = [
        word_count,
        unique_words,
        ttr,
        avg_word_len,
        sentence_count,
        avg_sentence_len,
        sentence_len_var,
        long_word_ratio,
        short_word_ratio,
        punct_density
    ]

    feat = scaler_feat.transform([feat])

    vec_full = np.hstack([vec, feat.flatten()]).reshape(1, -1)

    vec_scaled = best_scaler.transform(vec_full)

    pred = best_svr.predict(vec_scaled)[0]

    pred = np.clip(pred, 1, 9)
    pred = [round(s * 2) / 2 for s in pred]

    overall = round(np.mean(pred) * 2) / 2

    return {
        "Task_Response": pred[0],
        "Coherence_Cohesion": pred[1],
        "Lexical_Resource": pred[2],
        "Grammatical_Range_Accuracy": pred[3],
        "Overall": overall
    }

In [17]:
band_6 = """ The two maps illustrate the significant redevelopment  of an Australian zoo, its layout in 1960 with its current configuration.

Overall, the zoo has undergone a  substantial modernization, primarily characterized by the significant enhancement of its transportation infrastructure  and accessibility. The most striking developments are the establishment of two new entrances and the introduction of both a ferry service and a cable car system.

In 1960, the zoo was accessible via a northern harbour. The eastern sector featured a Picnic Area adjacent to a Seal Show. Centrally, enclosures for birds and chimpanzees were located near the Gift Shop, while the western area was designated for housing African and Australian animals.

By the present day, access has been diversified with a new Entrance and ferry terminal by the harbour, and a second Entrance in the south-west. A cable car now links this south-western entrance directly to the Picnic Area, and the original Gift Shop has been significantly expanded. Furthermore, the central enclosures for birds and chimpanzees have been replaced by African Animals, whose former western habitat now houses 'Wild animals'. The Seal Show and Picnic Area, however, have remained unchanged.
"""

band_8 = """The two maps illustrate the significant modifications made to a student residence hall between 2010 and the present day.

Overall, the primary development involved a substantial increase in accommodation capacity, resulting in more bedrooms. This expansion was achieved by repurposing and replacing former communal spaces, specifically the living room and the south garden.

In 2010, the residence hall comprised three study-bedrooms. A large living room was situated in the northern part of the building, with the kitchen located to the east. The property also featured two separate garden areas, one in the north and another in the south.

By the present day, the layout has been comprehensively altered. The former living room has been converted into an additional study-bedroom. Concurrently, the original study-bedroom in the west was reconfigured into two smaller study-bedrooms, and the south-eastern bedroom was modified to incorporate an ensuite bathroom. These changes raised the total bedroom count from three to five. Externally, the south garden was paved over to create a parking area, while the original kitchen was expanded to function as a combined kitchen and social area.
"""

print("Band 6 Prediction:", predict_task1_svr(band_6))
print("Band 8 Prediction:", predict_task1_svr(band_8))


Band 6 Prediction: {'Task_Response': 8.0, 'Coherence_Cohesion': 7.5, 'Lexical_Resource': 7.5, 'Grammatical_Range_Accuracy': 8.0, 'Overall': 8.0}
Band 8 Prediction: {'Task_Response': 8.0, 'Coherence_Cohesion': 7.5, 'Lexical_Resource': 7.0, 'Grammatical_Range_Accuracy': 7.5, 'Overall': 7.5}
